# Pipe Segment Visualization

PipeEndProfileAnalyzer?먯꽌 吏?뺥븳 媛쒖닔???멸렇癒쇳듃瑜?異붿젙?섍퀬, ???명듃遺곸뿉?쒕뒗 寃곌낵留??쒓컖?뷀빀?덈떎.

In [1]:
from pathlib import Path
import sys
import json
import importlib

import numpy as np
import open3d as o3d

PLUGIN_ROOT = Path.cwd()
if PLUGIN_ROOT.name != "poseDeterminator":
    PLUGIN_ROOT = Path("python/plugins/poseDeterminator").resolve()

if str(PLUGIN_ROOT) not in sys.path:
    sys.path.insert(0, str(PLUGIN_ROOT))

import PipeEndProfileAnalyzer as pipe_analyzer_module
pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
analyze_pipe_end_profiles = pipe_analyzer_module.analyze_pipe_end_profiles

PLUGIN_ROOT


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator')

## Settings

TARGET_SEGMENT_COUNT瑜??먰븯??吏곸꽑 諛곌? 援ш컙 ?섎줈 留욎텣 ???꾨옒 ?遺???ㅽ뻾?⑸땲??

In [2]:
PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.1.ply"
# PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.3_fill.ply"

SCALE = 1.0
VOXEL_SIZE = None
MAX_POINTS = 25000
TARGET_SEGMENT_COUNT = 6
MAX_SEGMENTS = TARGET_SEGMENT_COUNT
RANSAC_ITERATIONS = 50
SAMPLE_SIZE = 128
DISTANCE_THRESHOLD = None
PROFILE_SAMPLE_COUNT = 96
LOG_TIMING = True

assert PIPE_PATH.exists(), PIPE_PATH
PIPE_PATH


WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator/data/PIPE NO.1.ply')

## Analyze

吏?뺥븳 TARGET_SEGMENT_COUNT???꾨떖???뚭퉴吏 normal 湲곕컲 cylinder ?꾨낫 RANSAC??諛섎났?⑸땲??

In [3]:
result = analyze_pipe_end_profiles(
    PIPE_PATH,
    scale=SCALE,
    voxel_size=VOXEL_SIZE,
    max_points=MAX_POINTS,
    max_segments=MAX_SEGMENTS,
    target_segment_count=TARGET_SEGMENT_COUNT,
    ransac_iterations=RANSAC_ITERATIONS,
    sample_size=SAMPLE_SIZE,
    distance_threshold=DISTANCE_THRESHOLD,
    profile_sample_count=PROFILE_SAMPLE_COUNT,
    include_segment_points=True,
    log_timing=LOG_TIMING,
    random_seed=0,
)

summary = {
    "point_count": result["point_count"],
    "fit_point_count": result["fit_point_count"],
    "unassigned_fit_point_count": result["unassigned_fit_point_count"],
    "distance_threshold": result["distance_threshold"],
    "target_segment_count": result["target_segment_count"],
    "timing_sec": result["timing_sec"],
    "segments": [
        {
            "axis": seg["axis"],
            "radius": seg["radius"],
            "inlier_count": seg["inlier_count"],
            "rms_error": seg["rms_error"],
            "endpoints": seg["endpoints"],
        }
        for seg in result["segments"]
    ],
    "joint_endpoint_pairs": result["joint_endpoint_pairs"],
    "terminal_end_centers": [profile["center"] for profile in result["terminal_end_profiles"]],
}

print(json.dumps(summary, indent=2, ensure_ascii=False))


Analyzing pipe end profiles from d:\flame_robotics_drt\python\plugins\poseDeterminator\data\PIPE NO.1.ply...
  Total points: 500000
  Fit points: 25000
  bbox diagonal: 1818.969334
  Distance threshold: 9.094847
[timing] segment 0 iter 0 new best: 0.045s | score=6658, sample_points=477, normal_sample_sec=0.0008, normal_estimate_sec=0.0100, cylinder_sample_sec=0.0021, cluster_sec=0.0001, farthest_sec=0.0001, sphere_sample_sec=0.0008, fit_cylinder_sec=0.0295, total_sec=0.0436
[timing] segment 0 iter 2 new best: 0.041s | score=10234, sample_points=324, normal_sample_sec=0.0006, normal_estimate_sec=0.0113, cylinder_sample_sec=0.0012, cluster_sec=0.0001, farthest_sec=0.0000, sphere_sample_sec=0.0005, fit_cylinder_sec=0.0249, total_sec=0.0387
[timing] segment 0 iter 14 new best: 0.034s | score=10374, sample_points=391, normal_sample_sec=0.0007, normal_estimate_sec=0.0070, cylinder_sample_sec=0.0009, cluster_sec=0.0000, farthest_sec=0.0000, sphere_sample_sec=0.0003, fit_cylinder_sec=0.0236, t

## Visualize Segments

?됱긽蹂??먭뎔? 媛??멸렇癒쇳듃 inlier, 媛숈? ??wireframe? 異붿젙??理쒖쥌 ?ㅻ┛?붿엯?덈떎. 寃? ?먯? ?멸렇癒쇳듃濡??좊떦?섏? ?딆? fit point?낅땲??

In [4]:
def unit_vector(vector):
    vector = np.asarray(vector, dtype=float).reshape(3)
    norm = np.linalg.norm(vector)
    if norm <= 1e-12:
        raise ValueError(f"zero-length vector: {vector}")
    return vector / norm


def rotation_matrix_from_vectors(source, target):
    source = unit_vector(source)
    target = unit_vector(target)
    cross = np.cross(source, target)
    dot = float(np.clip(np.dot(source, target), -1.0, 1.0))
    cross_norm = np.linalg.norm(cross)
    if cross_norm <= 1e-12:
        if dot > 0.0:
            return np.eye(3)
        helper = np.array([1.0, 0.0, 0.0])
        if abs(source[0]) > 0.9:
            helper = np.array([0.0, 1.0, 0.0])
        axis = unit_vector(np.cross(source, helper))
        return o3d.geometry.get_rotation_matrix_from_axis_angle(axis * np.pi)
    axis = cross / cross_norm
    angle = np.arctan2(cross_norm, dot)
    return o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)


def make_point_cloud(points, color):
    points = np.asarray(points, dtype=float).reshape((-1, 3))
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.paint_uniform_color(color)
    return pcd


def make_sphere(center, radius, color):
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=24)
    sphere.translate(np.asarray(center, dtype=float).reshape(3))
    sphere.paint_uniform_color(color)
    return sphere


def make_line(points, color):
    points = np.asarray(points, dtype=float).reshape((-1, 3))
    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector(points)
    line.lines = o3d.utility.Vector2iVector([[0, 1]])
    line.paint_uniform_color(color)
    return line


def make_cylinder_wireframe(segment, color):
    endpoints = np.asarray(segment["endpoints"], dtype=float)
    axis_vec = endpoints[1] - endpoints[0]
    height = float(np.linalg.norm(axis_vec))
    if height <= 1e-12:
        return None
    cylinder = o3d.geometry.TriangleMesh.create_cylinder(
        radius=float(segment["radius"]),
        height=height,
        resolution=64,
        split=8,
    )
    cylinder.rotate(rotation_matrix_from_vectors([0.0, 0.0, 1.0], axis_vec), center=[0.0, 0.0, 0.0])
    cylinder.translate((endpoints[0] + endpoints[1]) * 0.5)
    lines = o3d.geometry.LineSet.create_from_triangle_mesh(cylinder)
    lines.paint_uniform_color(color)
    return lines


context_pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    context_pcd.scale(SCALE, center=np.zeros(3))
if VOXEL_SIZE is not None and VOXEL_SIZE > 0:
    context_pcd = context_pcd.voxel_down_sample(VOXEL_SIZE)
if len(context_pcd.points) > 12000:
    context_pcd = context_pcd.random_down_sample(12000 / len(context_pcd.points))
context_pcd.paint_uniform_color([0.78, 0.78, 0.78])

bbox_extent = context_pcd.get_axis_aligned_bounding_box().get_extent()
marker_radius = max(float(np.linalg.norm(bbox_extent)) * 0.006, 1e-6)
segment_colors = [
    [1.0, 0.15, 0.12],
    [0.1, 0.35, 1.0],
    [0.1, 0.75, 0.25],
    [0.75, 0.25, 1.0],
    [1.0, 0.75, 0.0],
]

geometries = [context_pcd]
for idx, segment_points in enumerate(result.get("segment_point_clouds", [])):
    color = segment_colors[idx % len(segment_colors)]
    points = np.asarray(segment_points, dtype=float)
    geometries.append(make_point_cloud(points, color))

    segment = result["segments"][idx]
    cylinder_wire = make_cylinder_wireframe(segment, color)
    if cylinder_wire is not None:
        geometries.append(cylinder_wire)

    endpoints = np.asarray(segment["endpoints"], dtype=float)
    geometries.append(make_line(endpoints, color))
    for endpoint in endpoints:
        geometries.append(make_sphere(endpoint, marker_radius, color))

unassigned_points = np.asarray(result.get("unassigned_fit_points", []), dtype=float)
if unassigned_points.size:
    geometries.append(make_point_cloud(unassigned_points, [0.05, 0.05, 0.05]))

print(f"segments: {len(result['segments'])}")
for idx, segment in enumerate(result["segments"]):
    print(
        f"S{idx}: inliers={segment['inlier_count']}, radius={segment['radius']:.6f}, rms={segment['rms_error']:.6f}"
    )

o3d.visualization.draw_geometries(geometries)

segments: 6
S0: inliers=10775, radius=34.432588, rms=3.078391
S1: inliers=7802, radius=33.881029, rms=3.716537
S2: inliers=2610, radius=34.426075, rms=4.043125
S3: inliers=1813, radius=32.920122, rms=4.465328
S4: inliers=621, radius=14.390156, rms=3.548692
S5: inliers=616, radius=22.138817, rms=3.900135
